In [2]:
%pwd

'c:\\Users\\bbhav\\OneDrive\\Desktop\\langchain\\Medical_Chatbot\\research'

In [1]:
print("OK")

OK


In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\bbhav\\OneDrive\\Desktop\\langchain\\Medical_Chatbot'

In [5]:
from langchain_community.document_loaders import PyPDFLoader

ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [2]:
def load_pdf_files(data):
    loader = DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [3]:
extracted_data = load_pdf_files("data")

NameError: name 'DirectoryLoader' is not defined

In [ ]:
from typing import List
from langchain.schema import Document
def filter_to_minimal_info(documents: List[Document]) -> List[Document]:
    minimal_docs: List[Document] = []
    for doc in documents:
        # Extract only the first 100 characters of the content
    src = doc.metadata.get("source")
   
     minimal_docs.append(Document(page_content=doc.page_content, metadata={"source": src}))
    return minimal_docs

In [ ]:
def text_split(minimal_info_docs) -> List[Document]:
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(minimal_docs)
    return text_chunks

In [ ]:
text_chunks = text_split(minimal_docs)
print(text_chunks[0])

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

embedding = download_embeddings()

In [ ]:
vector = embedding.embed_query("What is the purpose of this clinical trial?")
print(len(vector))

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY
pc = Pinecone(api_key=pinecone_api_key)


In [ ]:
pc

In [ ]:
from pinecone import ServerlessSpec

index_name = "medical_chatbot"

if not pc.has_index(index_name):
    pc.create_index(name=index_name, dimension=384, metric="cosine", serverless_spec=ServerlessSpec(cloud="aws", region="us-west-1"))
index = pc.index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks, 
    embedding=embedding, 
    index_name=index_name
    )

In [ ]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embedding)

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
retrieved_docs = retriever.invoke("What is the purpose of this clinical trial?")
print(retrieved_docs)

In [ ]:
from langchain_openai import ChatOpenAI
chatmodel = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.prompts import ChatPromptTemplate
from langchain.chain import create_stuff_documents_chain


In [ ]:
system_prompt = (
    "You are a helpful assistant that answers questions about clinical trials based on the retrieved documents. "
    "Use the following retrieved documents to answer the question. If you don't know the answer, say you don't know."
    "\n\n"
    "{context}"

)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [ ]:
question_answering_chain = create_stuff_documents_chain(chatmodel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answering_chain)